# RAGAS Testset Generator — OpenAI

Generate question/answer/context triples from Vietnamese company policy documents  
using RAGAS 0.2.x `TestsetGenerator` + OpenAI GPT-4o-mini (LLM) + `text-embedding-3-small` (embeddings).

**Documents**: 6 HR policy `.txt` files in `data/`  
**Output**: `testset_openai.csv` with columns `user_input`, `reference_contexts`, `reference`, `synthesizer_name`

> RAGAS 0.2.x renamed columns: `question`→`user_input`, `contexts`→`reference_contexts`, `ground_truth`→`reference`, `evolution_type`→`synthesizer_name`

> Compare with `ragas-generate-questions.ipynb` which uses Gemini.

## 1. Install Dependencies

Pin `ragas` to `0.2.x` for the `TestsetGenerator` API.  
**Restart the kernel after running this cell** if a different ragas version was previously loaded.

In [8]:
%pip install -q \
    "ragas>=0.2.0,<0.3.0" \
    "langchain-openai>=0.1.0" \
    "langchain-community>=0.2.0,<0.4.0" \
    "python-dotenv>=1.0.0"


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip3.13 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 1.4 MB/s  0:00:01 eta 0:00:01

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip3.13 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Environment Setup

LangChain OpenAI reads `OPENAI_API_KEY` natively — no mapping needed.

Create a `.env` file in this directory with:
```
OPENAI_API_KEY=sk-...
```

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path(".env"), override=False)

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError(
        "OPENAI_API_KEY is not set. "
        "Create a .env file in this directory with OPENAI_API_KEY=sk-..."
    )

print(f"OPENAI_API_KEY length: {len(os.environ['OPENAI_API_KEY'])} chars")

OPENAI_API_KEY length: 164 chars


## 3. Load Documents

Load all `.txt` files from `data/` with UTF-8 encoding (required for Vietnamese text).

In [3]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    "data",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True,
)

documents = loader.load()

print(f"\nLoaded {len(documents)} documents")
print(f"{'File':<45} {'Chars':>8}")
print("-" * 55)
for doc in sorted(documents, key=lambda d: d.metadata.get("source", "")):
    fname = Path(doc.metadata.get("source", "?")).name
    print(f"  {fname:<43} {len(doc.page_content):>8,}")

100%|██████████| 6/6 [00:00<00:00, 1532.91it/s]


Loaded 6 documents
File                                             Chars
-------------------------------------------------------
  chinh_sach_lam_viec_tu_xa.txt                  4,718
  chinh_sach_nghi_phep.txt                       4,140
  chinh_sach_phuc_loi.txt                        5,122
  quy_dinh_trang_phuc.txt                        4,566
  quy_trinh_xin_phep.txt                         4,690
  so_tay_nhan_vien.txt                           5,824


## 4. Initialize OpenAI LLM and Embeddings

- **LLM**: `gpt-4o-mini` — cost-effective and multilingual, good for Vietnamese question generation. Switch to `gpt-4o` for higher quality at higher cost.
- **Embeddings**: `text-embedding-3-small` (1536 dims) — multilingual, supports Vietnamese. Use `text-embedding-3-large` (3072 dims) for better semantic matching.

In [4]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

_llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.3,
)

_embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
)

generator_llm = LangchainLLMWrapper(_llm)
generator_embeddings = LangchainEmbeddingsWrapper(_embeddings)

print("LLM:", _llm.model_name)
print("Embeddings:", _embeddings.model)
print("Wrappers ready.")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LLM: gpt-4o
Embeddings: text-embedding-3-small
Wrappers ready.


## 5. Generate Testset

Runs `TestsetGenerator.generate_with_langchain_docs()` with `testset_size=10`.

Internally RAGAS:
1. Chunks the documents and builds an embedding-based knowledge graph
2. Synthesizes questions of mixed types (`simple`, `reasoning`, `multi_context`)
3. Generates ground-truth answers via GPT-4o-mini

**Allow 2–5 minutes** for the OpenAI API calls to complete.

In [5]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,  # note: param name is embedding_model, not embeddings
)

testset = generator.generate_with_langchain_docs(
    documents,
    testset_size=20,
)

print(f"\nGenerated {len(testset)} samples.")

Generating Samples: 100%|██████████| 21/21 [01:29<00:00,  4.25s/it]



Generated 21 samples.


## 6. Inspect the Generated Dataset

In [10]:
import pandas as pd

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 20)

df = testset.to_pandas()

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nQuestion types:\n{df['synthesizer_name'].value_counts().to_string()}\n")
df

Shape: (21, 4)
Columns: ['user_input', 'reference_contexts', 'reference', 'synthesizer_name']

Question types:
synthesizer_name
single_hop_specifc_query_synthesizer    7
multi_hop_abstract_query_synthesizer    7
multi_hop_specific_query_synthesizer    7



,user_input,reference_contexts,reference,synthesizer_name
0,CÔNG TY AGENTICHOUSE nghỉ phép sao?,[CHÍNH SÁCH NGHỈ PHÉP – CÔNG TY AGENTICHOUSE Phiên bản: 3.0 | Ban hành: 01/01/2024 | Phòng Nhân sự === CHƯƠNG 1: NGH...,"Nhân viên chính thức dưới 5 năm thâm niên có 12 ngày nghỉ phép năm, từ 5 đến dưới 10 năm có 14 ngày, và từ 10 năm tr...",single_hop_specifc_query_synthesizer
1,Wht r the cnsequences of nt fllwing the Nội quy Lao động?,"[CHƯƠNG 4: NGHỈ LỄ VÀ NGHỈ ĐẶC BIỆT === Điều 10. Nghỉ lễ tết Nhân viên được nghỉ đủ các ngày lễ, tết theo quy định c...",Nghỉ không phép hoặc vắng mặt không lý do sẽ bị trừ lương ngày tương ứng và bị ghi vào hồ sơ kỷ luật. Tái phạm 3 lần...,single_hop_specifc_query_synthesizer
2,What are the benefits provided by the PTI Care health insurance policy for employees at Agentic House?,[CHÍNH SÁCH PHÚC LỢI NHÂN VIÊN – CÔNG TY AGENTIC HOUSE Phiên bản: 2.3 | Ban hành: 01/01/2024 | Phòng Nhân sự === CHƯ...,The PTI Care health insurance policy at Agentic House provides the following benefits: For all official employees fr...,single_hop_specifc_query_synthesizer
3,Wht is Udemy used for in the training budget?,[CHƯƠNG 4: PHÁT TRIỂN NGHỀ NGHIỆP === Điều 9. Ngân sách đào tạo Mỗi nhân viên chính thức được cấp ngân sách đào tạo ...,Udemy is used for registering online courses as part of the annual training budget allocated to employees.,single_hop_specifc_query_synthesizer
4,What are the core services and mission of Agentic House JSC?,[SỔ TAY NHÂN VIÊN – CÔNG TY AGENTIC HOUSE Phiên bản: 4.0 | Ban hành: 01/01/2024 | Phòng Nhân sự === PHẦN 1: GIỚI THI...,"Agentic House JSC, established in 2015, specializes in providing enterprise software solutions, cloud computing serv...",single_hop_specifc_query_synthesizer
...,...,...,...,...
16,"What are the requirements and support provided for remote work, and how do they relate to the company's policies on ...","[<1-hop>\n\nPHẦN 3: QUY ĐỊNH LÀM VIỆC === 3.1. Giờ làm việc - Giờ làm việc chính thức: 08:00 – 17:30, thứ Hai đến th...",The requirements for remote work include using a company laptop or an approved device with necessary security softwa...,multi_hop_specific_query_synthesizer
17,How do the dress code regulations for the IT department at Agentic House relate to the requirements for remote work ...,[<1-hop>\n\nQUY ĐỊNH TRANG PHỤC CÔNG TY AGENTIC HOUSE Phiên bản: 2.1 | Ban hành: 01/01/2024 | Phòng Nhân sự === CHƯƠ...,The dress code regulations for the IT department at Agentic House allow employees to wear business casual attire fro...,multi_hop_specific_query_synthesizer
18,How does the Agentic House HRM system facilitate the process of requesting annual leave and managing overtime work?,"[<1-hop>\n\nPHẦN 3: QUY ĐỊNH LÀM VIỆC === 3.1. Giờ làm việc - Giờ làm việc chính thức: 08:00 – 17:30, thứ Hai đến th...",The Agentic House HRM system facilitates the process of requesting annual leave by allowing employees to log in to h...,multi_hop_specific_query_synthesizer
19,What are the responsibilities of the IT Department (Phòng CNTT) in ensuring secure remote work and what support does...,[<1-hop>\n\nCHÍNH SÁCH LÀM VIỆC TỪ XA (REMOTE WORK) – CÔNG TY AGENTICHOUSE Phiên bản: 1.2 | Ban hành: 01/03/2024 | P...,"The IT Department (Phòng CNTT) is responsible for approving company laptops or devices for remote work, ensuring the...",multi_hop_specific_query_synthesizer


## 7. Save to CSV

`contexts` is a list — JSON-encode it so it survives CSV round-tripping.  
`utf-8-sig` encoding adds a BOM so Excel opens Vietnamese text correctly.

In [ ]:
import json

OUTPUT_PATH = (
    Path(__file__).parent
    / "eval-sets/testset_openai.csv"
)


df_save = df.copy()
if "reference_contexts" in df_save.columns:
    df_save["reference_contexts"] = df_save["reference_contexts"].apply(
        lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, list) else x
    )

df_save.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"Saved {len(df_save)} rows → {OUTPUT_PATH.resolve()}")
print(f"File size: {OUTPUT_PATH.stat().st_size:,} bytes")
print("\nTo reload:")
print("  df = pd.read_csv('testset_openai.csv')")
print("  df['reference_contexts'] = df['reference_contexts'].apply(json.loads)")

Saved 21 rows → /Users/nguyenduy/Documents/nguyenduy-math/agentichouse/agentichouse/rag-projects/ragas-assessment/testset_openai.csv
File size: 126,824 bytes

To reload:
  df = pd.read_csv('testset_openai.csv')
  df['reference_contexts'] = df['reference_contexts'].apply(json.loads)
